# Część 1: Trening modelu

## Zadanie 1.1 — Przygotuj dane i wytrenuj model

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import pickle

np.random.seed(42)

# === PARAMETRY — zmień tutaj ===
N_NORMAL = 2000      # liczba normalnych transakcji
N_FRAUD  = 100       # liczba fraudów
# ===============================

# Normalne transakcje
normal = pd.DataFrame({
    'amount': np.random.lognormal(5, 1, N_NORMAL).clip(5, 5000),
    'is_electronics': np.random.binomial(1, 0.3, N_NORMAL),
    'tx_per_minute': np.random.poisson(3, N_NORMAL),
    'fraud': 0
})


# Fraudy
fraud = pd.DataFrame({
    'amount': np.random.uniform(2000, 9000, N_FRAUD),
    'is_electronics': np.random.binomial(1, 0.7, N_FRAUD),
    'tx_per_minute': np.random.poisson(8, N_FRAUD),
    'fraud': 1
})

df = pd.concat([normal, fraud], ignore_index=True).sample(frac=1, random_state=42)
print(f"Dataset: {len(df)} wierszy, fraud rate: {df['fraud'].mean():.1%}")

Dataset: 2100 wierszy, fraud rate: 4.8%


## Zadanie 1.2 — Podziel dane, wytrenuj, oceń

In [5]:
features = ['amount', 'is_electronics', 'tx_per_minute']
X = df[features]
y = df['fraud']

# 1. Podział na zbiór treningowy i testowy (80/20, stratify=y dla zachowania proporcji klas)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)

# 2. Utworzenie klasyfikatora RandomForest z 100 drzewami decyzyjnymi
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

# 3. Wygenerowanie raportu klasyfikacji (precision, recall, f1-score)
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

# 4. Eksport wytrenowanego modelu do pliku za pomocą biblioteki pickle
with open('fraud_model.pkl', 'wb') as f:
    pickle.dump(clf, f)
print("Model został pomyślnie wytrenowany i zapisany jako 'fraud_model.pkl'")

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       400
           1       1.00      1.00      1.00        20

    accuracy                           1.00       420
   macro avg       1.00      1.00      1.00       420
weighted avg       1.00      1.00      1.00       420

Model został pomyślnie wytrenowany i zapisany jako 'fraud_model.pkl'


# Część 2: FastAPI

## Zadanie 2.1 — Utwórz API serwujące model

In [3]:
%%file fraud_api.py
from fastapi import FastAPI
from pydantic import BaseModel
import pickle, numpy as np

app = FastAPI(title="Fraud Detection API")
model = pickle.load(open('fraud_model.pkl', 'rb'))

class Transaction(BaseModel):
    amount: float
    is_electronics: int
    tx_per_minute: int

@app.post("/score")
def score(tx: Transaction):
    # Przygotowanie cech do modelu (tablica 2D numpy)
    features = np.array([[tx.amount, tx.is_electronics, tx.tx_per_minute]])
    
    # Predykcja binarna (0 lub 1)
    pred = model.predict(features)[0]
    
    # Prawdopodobieństwo przynależności do klasy fraud (klasa 1)
    prob = model.predict_proba(features)[0][1]
    
    return {
        "is_fraud": bool(pred),
        "fraud_probability": float(prob)
    }

# Endpoint zdrowia (Zadanie z pracy domowej)
@app.get("/health")
def health_check():
    return {
        "status": "ok",
        "model_loaded": model is not None
    }

Writing fraud_api.py


Uruchom aplikację w terminalu Jupytera:
```bash
uvicorn fraud_api:app --host 0.0.0.0 --port 8001
```

## Zadanie 2.2 — Przetestuj API

In [6]:
import requests

# Test normalna transakcja
r_normal = requests.post("http://localhost:8001/score",
    json={"amount": 150, "is_electronics": 0, "tx_per_minute": 3})
print("Transakcja normalna:", r_normal.json())

# Test podejrzanej transakcji (amount=5500, is_electronics=1, tx_per_minute=12)
r_fraud = requests.post("http://localhost:8001/score",
    json={"amount": 5500, "is_electronics": 1, "tx_per_minute": 12})
print("Transakcja podejrzana:", r_fraud.json())

# Przetestowanie endpointu zdrowia
r_health = requests.get("http://localhost:8001/health")
print("Health check:", r_health.json())

Transakcja normalna: {'is_fraud': False, 'fraud_probability': 0.0}
Transakcja podejrzana: {'is_fraud': True, 'fraud_probability': 0.99}
Health check: {'status': 'ok', 'model_loaded': True}


# Część 3: Kafka + ML

## Zadanie 3.1 — Konsument z scoringiem ML

In [7]:
%%file ml_consumer.py
from kafka import KafkaConsumer, KafkaProducer
from datetime import datetime
import json, requests

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='ml-scoring',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

alert_producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

API_URL = "http://localhost:8001/score"

print("Konsument ML Scoring uruchomiony. Oczekiwanie na transakcje...\n")

for message in consumer:
    tx = message.value
    
    # 1. Wyciągnięcie cech zgodnych z wymaganiami modelu
    amount = tx.get('amount', 0.0)
    
    # Kategoria 'elektronika' to is_electronics = 1, w przeciwnym wypadku 0
    is_electronics = 1 if tx.get('category') == 'elektronika' else 0
    
    # Domyślna wartość tx_per_minute = 5, lub zliczana na podstawie innych kryteriów
    tx_per_minute = tx.get('tx_per_minute', 5)
    
    features = {
        "amount": amount,
        "is_electronics": is_electronics,
        "tx_per_minute": tx_per_minute
    }
    
    try:
        # 2. Odpytanie serwera API (FastAPI)
        response = requests.post(API_URL, json=features)
        result = response.json()
        
        is_fraud = result.get("is_fraud", False)
        prob = result.get("fraud_probability", 0.0)
        
        # 3. Reakcja na wynik scoringu
        if is_fraud:
            alert = {
                "transaction": tx,
                "fraud_probability": prob,
                "alert_timestamp": datetime.now().isoformat()
            }
            # Wyślij powiadomienie do tematu 'alerts'
            alert_producer.send('alerts', value=alert)
            print(f"🚨 ALERT ML! ID: {tx['tx_id']} | Kwota: {amount:.2f} PLN | Prawdopodobieństwo fraudu: {prob:.1%} | Sklep: {tx['store']}")
        else:
            print(f"✔️ Transakcja OK | ID: {tx['tx_id']} | Kwota: {amount:.2f} PLN | Prawdopodobieństwo: {prob:.1%}")
            
    except Exception as e:
        print(f"❌ Błąd podczas komunikacji z API: {e}")

Writing ml_consumer.py


## Zadanie 3.2 — Uruchom pipeline

W 3 terminalach:

1. `python producer.py` (z Ćw. 1)
2. `uvicorn fraud_api:app` --host 0.0.0.0 --port 8001
3. `python ml_consumer.py`

Obserwuj alerty.

# Praca domowa

1. Porównaj wyniki scoringu regułowego (Ćw. 1) vs ML — który lepiej wykrywa?
2. Dodaj endpoint `GET /health` do API.
3. Wypchnij do Git.